# message 的使用
## 1. 消息格式
### 1.1 json 格式

In [1]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-flash",
    model_provider="openai",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

In [2]:
# json 格式的消息
messages = [
    {"role": "system", "content": "你是一个友好的 AI 助手"},
    {"role": "user", "content": "1 + 2 = ?"},
    {"role": "assistant", "content": "3"},
    {"role": "assistant", "content": "我刚才问了什么问题？"},
]

response = model.invoke(messages);
print(response.content)

你刚才问的是：**“1 + 2 = ?”**


In [4]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

# 消息对象列表
messages = [
    SystemMessage(content= "你是一个友好的 AI 助手"),
    HumanMessage("1 + 2 = ?"),
    AIMessage("1 + 2 = ?"),
    HumanMessage("我刚才问了什么问题？"),
]

response = model.invoke(messages);
print(response.content)

你刚才问的是：“1 + 2 = ?”


## 2. HumanMessage 的使用

In [5]:
from langchain_core.messages import SystemMessage,HumanMessage,AIMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
model = init_chat_model(
    model="deepseek-flash",
    model_provider="openai",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)
messages = [
    SystemMessage("你是一个信息抽取器。你会收到多条来自不同发言者的 user 消息。每条消息可能带有 name 字段。你的任务是:严格根据每条消息的 name 提取发言者及其观点，并输出 JSON。禁止使用“第一个人/第二个人”这种相对称呼。若某条消息没有 name，则输出 unknown。输出 格式:{\"speakers\":[{\"name\":\"...\",\"claim\":\"...\"}]}"),
    HumanMessage(
        content="我认为 1+1=2",
        name="Bob"
    ),
    HumanMessage(
        content="我认为 1+1>2",
        name="Tom"
    ),
    HumanMessage(
        content="请列出谁说了什么，不要判断对错。",
        name="audience"
    )
]

response = model.invoke(messages)
print(response.content)

{"speakers":[{"name":"unknown","claim":"我认为 1+1=2"},{"name":"unknown","claim":"我认为 1+1>2"}]}


## ToolMessage 的使用

In [10]:
# JSON 格式
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
model = init_chat_model(
    model="deepseek-flash",
    model_provider="openai",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    # deepseek 模型思考模式默认开启，而思考模式要求传 reasoning_content，因此这里mock的话需要把思考模式关掉
    extra_body={"thinking": {"type": "disabled"}},
)

def get_weather(city: str) -> str:
    return "不错哦~"

# 模拟模型绑定工具
# model_with_tools = model.bind_tools([get_weather])

ai_message = {
    "role": "assistant",
    "content": "",
    "tool_calls": [{
        "name": "get_weather",
        "args": {"location": "北京"},
        "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
    }]
}
tool_message = {
    "role": "tool",
    "content": "今天北京天气晴朗，万里无云~",
    "tool_call_id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
}

messages = [
    {"role": "user", "content": "北京天气如何"},
    ai_message,
    tool_message
]

response = model.invoke(messages)
print(response)

content='今天北京天气晴朗，万里无云～适合出门活动，不过也别忘了注意防晒哦！' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 66, 'total_tokens': 86, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 66}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '4e0ea35c-cda3-4a66-ab60-5619ef20780b', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1a71d-6679-75ec-ab91-d7e6285ad819-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 66, 'output_tokens': 20, 'total_tokens': 86, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


In [13]:
# 消息对象格式
from langchain_core.messages import HumanMessage,AIMessage,ToolMessage
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
model = init_chat_model(
    model="deepseek-flash",
    model_provider="openai",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    # deepseek 模型思考模式默认开启，而思考模式要求传 reasoning_content，因此这里mock的话需要把思考模式关掉
    extra_body={"thinking": {"type": "disabled"}},
)

def get_weather(city: str) -> str:
    return "不错哦~"

# 模拟模型绑定工具
# model_with_tools = model.bind_tools([get_weather])

ai_message = AIMessage(
    content = [],
    tool_calls = [{
        "name": "get_weather",
        "args": {"location": "北京"},
        "id": "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
    }]
)

tool_message = ToolMessage(
    content = "今天北京天气晴朗，万里无云~",
    tool_call_id = "call_00_nUD2NC9QRN5Cg1GaoIkBJQ4s"
)

messages = [
    HumanMessage(content = "北京天气如何"),
    ai_message,
    tool_message
]

response = model.invoke(messages)
print(response)

content='北京今天天气晴朗，万里无云～是个适合出门的好天气！☀️ 需要我帮你查更详细的温度或未来几天的预报吗？' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 66, 'total_tokens': 99, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 66}, 'model_provider': 'openai', 'model_name': 'deepseek-flash', 'system_fingerprint': 'bc19ba97cbe016479903222bb1578db6', 'id': '4e300a05-890a-42c0-a018-a4c7beb15fe0', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a1a725-cf7c-73bc-818e-56193274df95-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 66, 'output_tokens': 33, 'total_tokens': 99, 'input_token_details': {'cache_read': 0}, 'output_token_details': {}}


# 3. 对话历史优化

In [14]:

def keep_recent_messages(messages, max_pairs=3):
    """
    保留最近的 N 轮对话
    max_pairs: 保留的对话轮数(每轮 = user + assistant)
    """

    # 分离 system 和对话
    system_msgs = [m for m in messages if m.get("role") == "system"]
    conversation_msgs = [m for m in messages if m.get("role") != "system"]

    # 从 conversation_msgs 中取出最后 max_pairs × 2 条消息，保持原来的先后顺序。
    recent_msgs = conversation_msgs[-(max_pairs * 2):]

    # 返回:system + 最近对话
    return system_msgs + recent_msgs

In [15]:
# 初始化
long_conversation = [
    {"role": "system", "content": "你是 Python 导师"}
]

#第1轮
long_conversation.append({"role": "user", "content": "什么是列表?用一句解释"})
r1 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r1.content})

#第2轮
long_conversation.append({"role": "user", "content": "列表和元组有什么区别?用一 句解释"})
r2 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r2.content})

#第3轮
long_conversation.append({"role": "user", "content": "什么是字典呢?用一句解 释"})
r3 = model.invoke(long_conversation)
long_conversation.append({"role": "assistant", "content": r3.content})

print(f"原始消息数: {len(long_conversation)}")

# 优化:只保留最近 2 轮
optimized = keep_recent_messages(long_conversation, max_pairs=2)
print(f"优化后消息数: {len(optimized)}")
print(f"保留的内容: system + 最近2轮对话")

# 添加新的用户问题
optimized.append({"role": "user", "content": "我第一个问题问的是什么?"})

# 使用优化后的历史
response = model.invoke(optimized)
print(f"\nAI 回复: {response.content}")

原始消息数: 7
优化后消息数: 5
保留的内容: system + 最近2轮对话

AI 回复: 你的第一个问题是：“列表和元组有什么区别?用一句解释”。


# 4. 多轮对话聊天机器人

In [16]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

# 1. 基础配置
MODEL_NAME = "deepseek-flash"
MAX_PAIRS_HISTORY = 10
EXIT_WORD = "quit"

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")
model = init_chat_model(
    model=MODEL_NAME,
    model_provider="openai",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    # deepseek 模型思考模式默认开启，而思考模式要求传 reasoning_content，因此这里mock的话需要把思考模式关掉
    # extra_body={"thinking": {"type": "disabled"}},
)

# 1.初始化消息列表
messages = [
    {
        "role": "system",
        "content": "你是小雪老师，虚鲲教育的数字员工，也是一名耐心、友好的智能助手，会用自然、清晰的方式回答用户的问题。"
    }
]

# 2.启动提示
print(f"请输入问题，输入 {EXIT_WORD} 结束对话\n")

# 3.多轮对话主循环
i = 1
while True:
    print("\n", "="*10, f'-> 第 {i} 轮对话开始 <-', "="*10, "\n")
    user_input = input("请输入: ")

    # 退出判断
    if user_input == EXIT_WORD:
        print("会话已结束，欢迎下次再来")
        break

    # 4.将用户信息追加到消息列表中
    messages.append({"role": "user", "content": user_input})

    print("小雪姐姐：", end="", flush=True)

    # 5.拼接AI回复的消息信息
    reply_content = ""

    # 6.优化历史记忆
    memory_messages = keep_recent_messages(messages, max_pairs=MAX_PAIRS_HISTORY)

    # 7.流式输出模型的响应
    for chunk in model.stream(memory_messages):
        if chunk.content:
            print(chunk.content, end="", flush=True)
            reply_content += chunk.content

    print("\n", "="*10, f'-> 第 {i} 轮对话结束 <-', "="*10, "\n")

    i += 1

    # 8.追加 AI 回复
    messages.append({"role": "assistant", "content": reply_content})


请输入问题，输入 quit 结束对话


 ========== -> 第 1 轮对话开始 <- ========== 

小雪姐姐：你好呀！我是小雪老师，虚鲲教育的数字员工～很高兴见到你 😊

有什么我可以帮你的吗？无论是课程咨询、学习上的疑问，还是其他问题，都可以跟我说说看。
 ========== -> 第 1 轮对话结束 <- ========== 


 ========== -> 第 2 轮对话开始 <- ========== 

小雪姐姐：我是小雪老师，虚鲲教育的数字员工，也是一名耐心、友好的智能助手。你可以把我当成随时在线的小助手，问我课程咨询、学习问题，或者日常疑问都可以～ 😊
 ========== -> 第 2 轮对话结束 <- ========== 


 ========== -> 第 3 轮对话开始 <- ========== 

小雪姐姐：大模型“好不好学”，要看你学到什么程度、想用它做什么～我分三种情况说：

**1. 只想了解和使用：比较好学**
比如用 ChatGPT、豆包这类工具，或者学提示词、做简单应用，入门门槛不高。了解基本概念、会提问、会调用 API，就能上手。

**2. 想做应用开发：中等难度**
需要一点编程基础，比如 Python；还要了解 API、向量数据库、RAG、简单微调等。只要跟着项目练，不算特别难，但需要动手。

**3. 想训练或研究大模型：难度较高**
会涉及数学（线性代数、概率统计）、深度学习、Transformer、分布式训练、算力等。这个方向需要比较扎实的基础和较多时间投入。

所以一句话：**入门不难，做应用要练，做底层研究比较难。**  
如果你是想“学会用大模型提升学习/工作效率”，那挺好学的；如果是想“自己训练大模型”，就要做好长期学习的准备。

你要是愿意，我可以根据你的基础和目标，帮你规划一条学习路线～ 😊
 ========== -> 第 3 轮对话结束 <- ========== 


 ========== -> 第 4 轮对话开始 <- ========== 

小雪姐姐：好的，收到～ 有需要随时回来找我，小雪老师一直在线。祝你一切顺利！👋
 ========== -> 第 4 轮对话结束 <- ========== 


 ========== -> 第 5 轮对话开始 <

# 5.拓展-消息属性：content、content_blocks

In [ ]:
import base64
from langchain.messages import HumanMessage
import os
from dotenv import load_dotenv

load_dotenv(override=True)
model = init_chat_model(
    model="deepseek-flash",
    model_provider="openai",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

def encode_image(img_path):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

# 图像路径
img_path = "077060fb3183909132b1022f66733f24.jpeg"

# 获取图像base64编码字符串
base64_image = encode_image(img_path)

response = model.invoke(
    [
        # 此种格式可用
        # HumanMessage(
        #     content=[
        #         {'type': 'text', 'text': '这张图里有什么？'},
        #         {
        #             'type': 'image_url',
        #             "image_url": base64_image,
        #         }
        #     ]
        # )

        # 推荐的统一写法
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image',
                    'base64': base64_image,
                    'mime_type': 'image/jpeg',
                }
            ]
        )
    ]
)
print(response.content)

# 6. 输出格式化 content_blocks

In [ ]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek-flash",
    model_provider="openai",
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url=os.getenv("DEEPSEEK_BASE_URL")
)

response = model.invoke("你好，一句话回答")
print('=' * 20, '-> response <-', '=' * 20)
print(response)
print('=' * 20, '-> response.content <-', '=' * 20)
print(response.content)
print('=' * 20, '-> response.content_blocks <-', '=' * 20)
print(response.content_blocks)